# SIM V3 · Outdoor · Phase B + C + D — one run
**Directional** outdoor surrogate driven by **real base stations on real buildings** (`fw_bs_catalog`): building-mounted panel/sector sources, 10-channel input (the 9 material/Tx/freq channels + a directivity channel), held-out real station (**Forte Hall**) for g2. Same one-session, local-disk, CuPy+AMP flow as indoor. Pick a **GPU runtime**.

### 1 · Clone the repo + mount Drive

In [ ]:
# /content is wiped on every runtime restart, so (re)clone the code here.
import os, getpass
REPO_ROOT = '/content/indoor-walk-test'
if not os.path.isdir(os.path.join(REPO_ROOT, 'Physics Engine')):
    tok = getpass.getpass('GitHub token (repo read): ')
    os.system(f'git clone --depth 1 https://{tok}@github.com/cgm2179/indoor-walk-test.git "{REPO_ROOT}"')
try:
    from google.colab import drive; drive.mount('/content/drive')   # only for the final save
except ModuleNotFoundError:
    pass
print('repo present:', os.path.isdir(os.path.join(REPO_ROOT, 'Physics Engine', '2D', 'SIM V3')))

### 2 · Put SIM V3 on the path

In [ ]:
import sys, os
SIMV3 = os.path.join(REPO_ROOT, 'Physics Engine', '2D', 'SIM V3')
assert os.path.exists(os.path.join(SIMV3, '_bootstrap.py')), f'clone missing — re-run cell 1: {SIMV3}'
sys.path.insert(0, SIMV3); os.chdir(SIMV3)
import torch; print('SIM V3 =', SIMV3, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())

### 3 · Build the georeferenced NoMa city grid (from the committed OBJ)
The city grid is gitignored (~360 MB), so build it here from `Data/models/NoMa_DC/NoMa_DC_buildings.obj` (~1–3 min, once per runtime). `load_georef_city` injects the FCC-HQ anchor in-memory, so no separate patch step.

In [ ]:
import subprocess, sys, _bootstrap as B
grid_npy = B.CITY_DIR / 'material_grid.npy'
if not grid_npy.exists():
    subprocess.run([sys.executable, str(B.SIM3D / 'voxelize_city.py'), '--no-preview'],
                   cwd=REPO_ROOT, check=True)                 # add --bbox-lonlat -77.0144 38.900 -77.0005 38.908 to crop/speed up
print('city grid ready:', grid_npy.exists(), '->', grid_npy)

### 4 · Parameters (cellular bands with real stations; Forte Hall held out)

In [ ]:
BANDS   = ['LTE_B71_617', 'LTE_B13_751', 'LTE_B2_1960', 'NR_n41_2506']   # cellular donors in the catalog
HOLDOUT = 'BS7_forte_hall'                     # held-out real station for the g2 gate
OUT   = '/content/fw_data_bs'                  # LOCAL disk (fast, reliable)
CKPT  = '/content/fw_bs.pt'
EPOCHS, BASE, BATCH = 80, 32, 16
BOXES_PER_STATION = 48
MAX_CELLS = 6_000_000                          # A100: bigger band-adaptive regions; on T4 use ~1_600_000

### 5 · GPU acceleration (CuPy) — accelerates the base-station FDTD too
Same cell as indoor: it patches `fw_dataset._run_field`, which `fw_bs_catalog.bs_field` calls, so outdoor generation runs on the GPU unchanged.

In [ ]:
# --- GPU FDTD (CuPy): reuse FullWaveScene setup, run only the time loop on GPU ---
import numpy as np, math
import fw_dataset
from fullwave2d import FullWaveScene

C0 = 299_792_458.0
_cpu_run_field = fw_dataset._run_field          # keep original (fallback + parity check)

USE_GPU = False
try:
    import cupy as cp
    USE_GPU = cp.cuda.runtime.getDeviceCount() > 0
except Exception:
    try:  # Colab GPU runtime usually ships CuPy; install the CUDA-12 wheel if not
        import subprocess, sys
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'cupy-cuda12x'], check=True)
        import cupy as cp
        USE_GPU = cp.cuda.runtime.getDeviceCount() > 0
    except Exception as e:
        print('CuPy unavailable -> staying on CPU:', e)

def _laplacian_gpu(u, inv_h2):                  # matches Spatial_Physics.laplacian (np.roll)
    lap = cp.zeros_like(u)
    for ax in range(u.ndim):
        lap += cp.roll(u, 1, axis=ax) + cp.roll(u, -1, axis=ax)
    lap -= 2.0 * u.ndim * u
    return lap * inv_h2

def _run_field_gpu(classes, h, tx_ij, f_mhz, crossings):
    sim = FullWaveScene(classes, h, f_mhz, tx_ij, source='cw')     # identical CPU setup
    steps = int(round(crossings * max(classes.shape) * h / C0 / sim.dt))
    dt, f0 = sim.dt, sim.f0
    omega = 2.0 * np.pi * f0
    inv_h2 = float(sim.inv_h2)
    u      = cp.asarray(sim.u)                  # move only the fields the loop touches
    u_prev = cp.asarray(sim.u_prev)
    cdt2   = cp.asarray(sim.cdt2)
    inv1pa = cp.asarray(sim._inv1pa)
    _1ma   = cp.asarray(sim._1ma)
    damp   = cp.asarray(sim.damp)
    rigid  = cp.asarray(sim.rigid)
    src    = sim.src_idx
    warmup = int(0.6 * steps)                   # same as _run_field's simulate() call
    period_steps = max(1, int(round((1.0 / f0) / dt)))
    win_start = max(warmup, steps - 2 * period_steps)             # phasor_periods = 2
    acc = cp.zeros(u.shape, cp.complex128); n_win = 0
    for k in range(steps):                      # mirrors FullWaveScene.step() exactly
        lap = _laplacian_gpu(u, inv_h2)
        u_next = (2.0 * u - _1ma * u_prev + cdt2 * lap) * inv1pa
        u_next[src] += math.sin(omega * (k * dt))                # cw() soft source, t=k*dt
        u_next[rigid] = 0.0                     # perfect reflectors
        u_next *= damp                          # absorbing sponge
        u_prev = u * damp
        u = u_next
        if k >= win_start:                      # on-the-fly single-freq DFT (u at (k+1)dt)
            acc += u * complex(np.exp(-1j * omega * (k + 1) * dt))
            n_win += 1
    if not bool(cp.isfinite(u).all()):
        raise FloatingPointError('field blew up (GPU)')
    return cp.asnumpy((2.0 / max(n_win, 1)) * acc)               # complex phasor U, on CPU

if USE_GPU:
    fw_dataset._run_field = _run_field_gpu       # generate() -> _indoor_field -> this
    name = cp.cuda.runtime.getDeviceProperties(0)['name'].decode()
    # parity vs CPU on a tiny field so you can trust the port
    rng = np.random.default_rng(0)
    test = ((rng.random((96, 96)) < 0.12).astype(np.int8) * 2)   # sparse concrete
    Ucpu = _cpu_run_field(test, 0.06, (48, 48), 617.0, 1.2)
    Ugpu = _run_field_gpu(test, 0.06, (48, 48), 617.0, 1.2)
    rel = float(np.abs(Ugpu - Ucpu).max() / (np.abs(Ucpu).max() + 1e-30))
    print(f'GPU FDTD ON -> {name} | parity max|dU|/|U| = {rel:.2e} (want < 1e-6)')
else:
    print('GPU FDTD OFF -> CPU _run_field (pick a GPU runtime for the speed-up).')


### 6 · Phase B — generate from real base stations (directional FDTD)

In [ ]:
import fw_bs_catalog as fbs
fbs.generate_bs(BANDS, holdout_site=HOLDOUT, boxes_per_station=BOXES_PER_STATION,
                out_dir=OUT, max_cells=MAX_CELLS, seed=1)

### 7 · Phase C — train the U-Net surrogate (AMP, auto 10-channel)

In [ ]:
import glob, fw_unet2d
print(len(glob.glob(OUT + '/shard_*.npz')), 'shards at', OUT)
model, best = fw_unet2d.train(OUT, epochs=EPOCHS, base=BASE, bs=BATCH, out=CKPT)
print('best val_mse =', best)

### 8 · Phase D — g2 on the held-out real station (Forte Hall) + ONNX export

In [ ]:
!pip -q install onnx onnxruntime
import fw_export
model = fw_unet2d.load_model(CKPT)
print(fbs.validate_bs(model, BANDS, holdout_site=HOLDOUT, max_cells=MAX_CELLS))
fw_export.export_onnx(model.cpu(), fw_export.WEB / 'fw_bs.onnx'); print('exported fw_bs.onnx')

### 9 · Directional surrogate coverage per band (no FDTD)

In [ ]:
import numpy as np, matplotlib.pyplot as plt
grid, man, sts = fbs.load_stations(BANDS)
dev = fw_unet2d.pick_device(); model = model.to(dev)
by_band = {}
for s in fbs._dedupe_by_site(sts): by_band.setdefault(s['band'], s)   # one station per band
bands = list(by_band)
fig, axes = plt.subplots(1, len(bands), figsize=(4.5*len(bands), 4), squeeze=False)
for ax, band in zip(axes.flat, bands):
    st = by_band[band]
    p = fbs.bs_region(grid, man, st, npw=8.0, max_cells=MAX_CELLS)         # region only, no FDTD
    U = fbs._tiled_predict_bs(model, p.classes, p.tx_idx, p.h_m, st['f_mhz'], st['boresight'])
    env = 20*np.log10(np.abs(U)+1e-12); env -= env.max()
    im = ax.imshow(env.T, origin='lower', cmap='viridis', vmin=-45, vmax=0,
                   extent=[0, p.extent_m[0], 0, p.extent_m[1]])
    ax.plot(p.tx_idx[0]*p.h_m, p.tx_idx[1]*p.h_m, 'r*', ms=10)
    ax.set_title(f"{st['site']} · {band} · brg {st['boresight']:.0f}°")
    fig.colorbar(im, ax=ax, shrink=0.7)
fig.suptitle('Outdoor surrogate coverage (directional, no FDTD) — |U| dB'); fig.tight_layout(); plt.show()

### 10 · Save to Drive — flush + verify

In [ ]:
import shutil, glob
from google.colab import drive
shutil.copytree(OUT, '/content/drive/MyDrive/fw_data_bs', dirs_exist_ok=True)
shutil.copy(CKPT, '/content/drive/MyDrive/fw_bs.pt')
drive.flush_and_unmount(); drive.mount('/content/drive')
print(len(glob.glob('/content/drive/MyDrive/fw_data_bs/shard_*.npz')), 'shards on Drive (verified)')